# Data Cleaning
Loading the raw dataset and preparing it for feature engineering.
This includes dropping irrelevant columns, fixing data types, and handling missing values.

In [1]:
import pandas as pd
import numpy as np

# load raw dataset
df = pd.read_csv('../data/raw/Telco_customer_churn.csv')

print("Original Shape:", df.shape)
df.head(3)

Original Shape: (7043, 33)


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved


### 1. Drop Irrelevant Columns
Removing columns that are either identifiers, location-based, or redundant for modeling.

In [11]:
# saving churn reason separately before dropping - will use in retention engine later
churn_reason_df = df[['Churn Reason', 'Churn Value']].copy()
churn_reason_df.to_csv('../data/processed/churn_reasons.csv', index=False)
print("Churn reasons saved separately for retention engine")

# dropping columns that won't help in modeling
cols_to_drop = ['CustomerID', 'Count', 'Country', 'State', 'City', 
                'Zip Code', 'Lat Long', 'Latitude', 'Longitude',
                'Churn Label', 'Churn Score', 'Churn Reason']

df = df.drop(columns=cols_to_drop)

print("\nShape after dropping columns:", df.shape)
print("\nRemaining Columns:", df.columns.tolist())

Churn reasons saved separately for retention engine

Shape after dropping columns: (7043, 21)

Remaining Columns: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value', 'CLTV']


### 2. Fix Data Types
Ensuring all columns have correct data types before any further processing.

In [12]:
# fixing Total Charges - stored as string, needs to be numeric
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

# checking data types now
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

Data Types:
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges        float64
Churn Value            int64
CLTV                   int64
dtype: object

Missing Values:
Gender                0
Senior Citizen        0
Partner               0
Dependents            0
Tenure Months         0
Phone Service         0
Multiple Lines        0
Internet Service      0
Online Security       0
Online Backup         0
Device Protection     0
Tech Support          0
Streaming TV          0
Streaming Movies      0
Contract    

### 3. Handle Missing Values
Only 11 missing values in Total Charges - filling with median since it's a small number.

In [13]:
# checking which rows have missing Total Charges
print("Rows with missing Total Charges:")
print(df[df['Total Charges'].isnull()][['Tenure Months', 'Monthly Charges', 'Total Charges']])

# these are likely new customers with 0 tenure - filling with 0
df['Total Charges'] = df['Total Charges'].fillna(0)

print("\nMissing values after fix:", df['Total Charges'].isnull().sum())

Rows with missing Total Charges:
      Tenure Months  Monthly Charges  Total Charges
2234              0            52.55            NaN
2438              0            20.25            NaN
2568              0            80.85            NaN
2667              0            25.75            NaN
2856              0            56.05            NaN
4331              0            19.85            NaN
4687              0            25.35            NaN
5104              0            20.00            NaN
5719              0            19.70            NaN
6772              0            73.35            NaN
6840              0            61.90            NaN

Missing values after fix: 0


### 4. Clean Column Values
Some categorical columns have 'No internet service' or 'No phone service' 
which basically means 'No' - simplifying these for cleaner modeling.

In [16]:
# replacing 'No internet service' and 'No phone service' with 'No'
# they mean the same thing for modeling purposes
cols_to_fix = ['Multiple Lines', 'Online Security', 'Online Backup', 
               'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies']

for col in cols_to_fix:
    df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})

# verify
print("Unique values after fix:")
for col in cols_to_fix:
    print(f"{col}: {df[col].unique()}")

Unique values after fix:
Multiple Lines: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
Online Security: <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
Online Backup: <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
Device Protection: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
Tech Support: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
Streaming TV: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
Streaming Movies: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


### 5. Save Cleaned Dataset
Saving the cleaned data to processed folder for feature engineering.

In [17]:
# saving cleaned dataset
df.to_csv('../data/processed/telco_cleaned.csv', index=False)

print("Cleaned dataset saved successfully.")
print("Shape:", df.shape)
print("\nFinal Columns:", df.columns.tolist())

Cleaned dataset saved successfully.
Shape: (7043, 21)

Final Columns: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value', 'CLTV']
